# Tokenization & Byte-Pair Encoding

The tokenizer is the first thing in the stack and the one people think about least. It is
also **frozen at the start of pre-training** — every parameter the model learns is
downstream of it, so a tokenizer mistake cannot be fixed later without retraining from
scratch.

It decides how many tokens your data costs, how well the model handles arithmetic and
code, and how badly it is penalised for being in a language other than English. This
notebook builds a BPE tokenizer from nothing in pure Python, then measures the properties
that actually matter.

First topic in the [Pre-training](scaling-laws.ipynb) track.

## 1. What & Why

A language model works on integers, not text. The tokenizer is the bijection between
them, and there are three broad choices:

- **Characters/bytes** — tiny vocabulary, no unknown tokens ever, but sequences are very
  long, and since attention is quadratic in sequence length that is expensive.
- **Words** — short sequences, but an unbounded vocabulary, a hard unknown-word problem,
  and no way to relate `run` to `running`.
- **Subwords** (BPE, WordPiece, Unigram) — the compromise everything uses. Common words
  get one token, rare words decompose into pieces, and nothing is ever unknown.

**Byte-Pair Encoding** is a compression algorithm repurposed: start from bytes, and
repeatedly merge the most frequent adjacent pair into a new token. Run it `k` times and
you have a vocabulary of `256 + k` tokens whose merges are ordered by usefulness.

**Why the choice is consequential and irreversible:**

- **Cost.** Tokens are the unit of compute *and* of billing. A tokenizer that needs 30%
  more tokens for your data makes training and inference 30% more expensive.
- **Capability.** How numbers are split determines how hard arithmetic is. How
  indentation is split determines how well the model writes Python.
- **Fairness.** The same sentence in English and in Telugu can differ by 5–10× in token
  count under an English-tuned tokenizer, which means worse quality *and* higher cost for
  those users.

## 2. Mental Model

**A compression codebook, learned from your corpus and then frozen for life.**

BPE is literally a compression algorithm, and the useful intuitions come from that:

- **It optimises for average length on the training corpus.** Text resembling that corpus
  compresses well; text unlike it fragments. This is why a tokenizer trained on English
  web text handles code and other languages badly — not out of neglect, but because they
  were not in the objective.
- **Merges are ordered and applied greedily.** Merge #1 is the most useful; merge #40,000
  is some specific rare string. At encode time they are applied in learned order, which
  is what makes encoding deterministic.
- **The codebook is frozen when training starts.** Adding a token later means a randomly
  initialised embedding row in a trained model. There are ways to patch this, but none of
  them are as good as having got it right.

The one-sentence version: **the tokenizer is a lossy summary of what your training corpus
looked like, baked permanently into the model's input layer.**

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Vocabulary size** | Number of distinct tokens. Trades sequence length against embedding/softmax cost. Modern LLMs use ~32k–256k. |
| **Merge rules** | The ordered list BPE learns. Encoding applies them in order; the order *is* the tokenizer. |
| **Byte-level BPE** | Start from the 256 bytes rather than Unicode characters. Guarantees no unknown token for any input. What GPT-2 onward use. |
| **Pre-tokenization** | Splitting on whitespace/punctuation *before* BPE, so merges never cross word boundaries. A regex, and it matters more than it looks. |
| **Fertility** | Tokens per word. The compression ratio, and the fairness metric. |
| **WordPiece** | BERT's variant: merges by likelihood gain rather than raw frequency. |
| **Unigram LM** | SentencePiece's default: start large and prune, with a probabilistic model. Supports sampling multiple segmentations. |
| **Subword regularisation** | Training with sampled alternative segmentations for robustness. |
| **Special tokens** | `<|endoftext|>`, chat-template markers, tool tokens. Reserved and never produced by merges. |
| **Number tokenization** | Whether digits are split individually or merged. Directly affects arithmetic ability. |
| **Continuation marker** | How the encoder records that a token starts a word (`Ġ` in GPT-2, `##` in BERT). |

## 4. Setup

Pure standard library — the whole algorithm is about forty lines, and writing it is the
fastest way to stop finding tokenizers mysterious.

In [1]:
# %pip install numpy      # only for the fertility table at the end

import re
from collections import Counter

print("standard library only")

standard library only


## 5. Worked Examples

### Example 1 — train a byte-level BPE from scratch

The complete algorithm: count adjacent pairs, merge the most frequent, repeat.

In [2]:
_BASE = (
    "the cat sat on the mat. the cat ate the rat. "
    "a cat and a rat sat on a mat. the rat ran. "
    "the fat cat sat. the cat ran fast. cats and rats. "
)
# A second, more varied block so the merge sweep in Example 3 has vocabulary to chew on
# rather than exhausting after a couple of dozen merges.
_VARIED = (
    "language models learn statistical patterns from large text corpora. "
    "tokenization determines how many tokens a document costs to process. "
    "researchers measure compression, fertility, and downstream performance. "
    "training requires substantial computational resources and careful engineering. "
    "evaluation benchmarks should measure generalization rather than memorization. "
)
CORPUS = _BASE * 40 + _VARIED * 40

PRE_TOKEN_RE = re.compile(r"\s?\w+|\s?[^\s\w]+|\s+")

def pre_tokenize(text):
    '''Split before BPE so merges never span a word boundary. The leading space is kept
    ON the word, which is how GPT-2 distinguishes " cat" from "cat".'''
    return PRE_TOKEN_RE.findall(text)

def train_bpe(text, num_merges):
    # every word as a tuple of byte-values, with its frequency
    words = Counter(pre_tokenize(text))
    splits = {w: tuple(w.encode("utf-8")) for w in words}
    vocab = {i: bytes([i]) for i in range(256)}
    merges = []

    for step in range(num_merges):
        pairs = Counter()
        for w, freq in words.items():
            s = splits[w]
            for i in range(len(s) - 1):
                pairs[(s[i], s[i + 1])] += freq
        if not pairs:
            break
        best, count = pairs.most_common(1)[0]
        new_id = 256 + step
        vocab[new_id] = vocab[best[0]] + vocab[best[1]]
        merges.append((best, new_id))
        # apply the merge everywhere
        for w in list(splits):
            s, out, i = splits[w], [], 0
            while i < len(s):
                if i < len(s) - 1 and (s[i], s[i + 1]) == best:
                    out.append(new_id); i += 2
                else:
                    out.append(s[i]); i += 1
            splits[w] = tuple(out)
    return vocab, merges

vocab, merges = train_bpe(CORPUS, num_merges=40)

print("the first 12 merges learned, in order:")
for rank, (pair, new_id) in enumerate(merges[:12]):
    a = vocab[pair[0]].decode("utf-8", "replace")
    b = vocab[pair[1]].decode("utf-8", "replace")
    print(f"  {rank + 1:2d}. {a!r:>8} + {b!r:<8} -> {vocab[new_id].decode('utf-8', 'replace')!r}")

print(f"\nvocabulary: 256 bytes + {len(merges)} merges = {256 + len(merges)} tokens")
print("\nThe order is the whole tokenizer. Frequent pairs merge first, so the early")
print("merges are common letter pairs and the later ones are whole frequent words.")

the first 12 merges learned, in order:
   1.      'a' + 't'      -> 'at'
   2.      ' ' + 't'      -> ' t'
   3.      ' ' + 'c'      -> ' c'
   4.      'a' + 'n'      -> 'an'
   5.      ' ' + 'r'      -> ' r'
   6.      'h' + 'e'      -> 'he'
   7.      'o' + 'n'      -> 'on'
   8.      ' ' + 'm'      -> ' m'
   9.      'e' + 's'      -> 'es'
  10.     ' t' + 'he'     -> ' the'
  11.     ' c' + 'at'     -> ' cat'
  12.      ' ' + 's'      -> ' s'

vocabulary: 256 bytes + 40 merges = 296 tokens

The order is the whole tokenizer. Frequent pairs merge first, so the early
merges are common letter pairs and the later ones are whole frequent words.


### Example 2 — encode with the learned merges, and see the fragmentation

Encoding applies merges in the order they were learned. Words resembling the training
corpus compress to few tokens; anything else falls back toward bytes.

In [3]:
def encode(text, merges):
    rank = {pair: i for i, (pair, _) in enumerate(merges)}
    new_id_of = {pair: nid for pair, nid in merges}
    out = []
    for word in pre_tokenize(text):
        seq = list(word.encode("utf-8"))
        while len(seq) > 1:
            # the merge with the LOWEST rank (learned earliest) wins
            best, best_rank, best_i = None, len(rank) + 1, -1
            for i in range(len(seq) - 1):
                r = rank.get((seq[i], seq[i + 1]), None)
                if r is not None and r < best_rank:
                    best, best_rank, best_i = (seq[i], seq[i + 1]), r, i
            if best is None:
                break
            seq[best_i:best_i + 2] = [new_id_of[best]]
        out.extend(seq)
    return out

def show(text):
    ids = encode(text, merges)
    pieces = [vocab[i].decode("utf-8", "replace") for i in ids]
    return ids, pieces

for text in ["the cat sat", "the rat ran fast", "a catastrophic mistake", "zygote"]:
    ids, pieces = show(text)
    print(f"{text!r:26} {len(ids):2d} tokens  {pieces}")

print("\n'the cat sat' -- all in-distribution, so it compresses hard.")
print("'catastrophic' shares a prefix with 'cat' and then fragments into bytes.")
print("'zygote' never appeared, so it is essentially character-level.")
print("\nThis is the failure mode behind every 'why is the model bad at X' where X is")
print("rare in the training corpus: X is not just unfamiliar, it is also SHREDDED into")
print("many low-information tokens before the model ever sees it.")

'the cat sat'               4 tokens  ['t', 'he', ' cat', ' sat']
'the rat ran fast'          7 tokens  ['t', 'he', ' rat', ' ran', ' f', 'a', 'st']
'a catastrophic mistake'   16 tokens  ['a', ' cat', 'a', 'st', 'r', 'o', 'p', 'h', 'i', 'c', ' m', 'i', 'st', 'a', 'k', 'e']
'zygote'                    6 tokens  ['z', 'y', 'g', 'o', 't', 'e']

'the cat sat' -- all in-distribution, so it compresses hard.
'catastrophic' shares a prefix with 'cat' and then fragments into bytes.
'zygote' never appeared, so it is essentially character-level.

This is the failure mode behind every 'why is the model bad at X' where X is
rare in the training corpus: X is not just unfamiliar, it is also SHREDDED into
many low-information tokens before the model ever sees it.


### Example 3 — vocabulary size is a real trade-off, not "bigger is better"

More merges means shorter sequences and a larger embedding/softmax matrix. The compute
cost moves in both directions at once.

In [4]:
SAMPLE = "the cat sat on the mat and the rat ran fast past a fat cat "

print(f"{'merges':>7} {'vocab':>7} {'tokens':>7} {'chars/token':>12} "
      f"{'embed params':>14} {'attn cost':>11}")
D_MODEL = 4096
for n_merges in (0, 10, 40, 100, 300):
    v, m = train_bpe(CORPUS, n_merges)
    ids = encode(SAMPLE, m)
    vocab_size = 256 + len(m)
    embed = vocab_size * D_MODEL
    attn = len(ids) ** 2          # attention is quadratic in sequence length
    print(f"{n_merges:>7} {vocab_size:>7} {len(ids):>7} {len(SAMPLE)/len(ids):12.2f} "
          f"{embed:14,d} {attn:11,d}")

print("\nMore merges keep shortening the sequence -- with diminishing returns, because")
print("each additional merge is rarer than the last. Halving the sequence length cuts")
print("attention cost by ~4x, since attention is quadratic -- but the embedding and")
print("output softmax matrices grow")
print("linearly with the vocabulary, and at inference the softmax over vocabulary is")
print("computed at EVERY step.")
print("\nThat is the real trade: sequence length (quadratic) against vocabulary")
print("(linear, but paid every token). Real tokenizers land at 32k-256k because that")
print("is roughly where the two costs balance for natural language -- and the optimum")
print("shifts with corpus and model size, which is why it is a tuned choice.")

 merges   vocab  tokens  chars/token   embed params   attn cost
      0     256      59         1.00      1,048,576       3,481
     10     266      38         1.55      1,089,536       1,444
     40     296      22         2.68      1,212,416         484
    100     356      19         3.11      1,458,176         361
    300     465      18         3.28      1,904,640         324

More merges keep shortening the sequence -- with diminishing returns, because
each additional merge is rarer than the last. Halving the sequence length cuts
attention cost by ~4x, since attention is quadratic -- but the embedding and
output softmax matrices grow
linearly with the vocabulary, and at inference the softmax over vocabulary is
computed at EVERY step.

That is the real trade: sequence length (quadratic) against vocabulary
(linear, but paid every token). Real tokenizers land at 32k-256k because that
is roughly where the two costs balance for natural language -- and the optimum
shifts with corpus an

### Example 4 — the two failure modes everyone hits: numbers and other languages

Both follow directly from "BPE optimises average length on the training corpus".

In [5]:
# Train on text where some numbers are common and others never appear.
NUM_CORPUS = ("in 2020 and 2021 and 2022 the value was 100 or 200 or 300 . " * 60)
nv, nm = train_bpe(NUM_CORPUS, num_merges=60)

print("NUMBERS -- how a number splits depends on whether it was frequent in training:\n")
for num in ["2020", "2021", "2023", "100", "137", "4271"]:
    ids = encode(num, nm)
    pieces = [nv[i].decode("utf-8", "replace") for i in ids]
    print(f"  {num:>6} -> {len(ids)} token(s): {pieces}")

print("\n'2020' is one token because it was frequent; '2023' was never seen and splits")
print("differently. So '2020' and '2023' have completely unrelated representations, and")
print("the model cannot see that they are adjacent years.")
print("\nThis is why arithmetic is hard for LLMs and why several modern tokenizers force")
print("digits to split individually (or in fixed 3-digit groups): a CONSISTENT")
print("decomposition is worth more than a short one.\n")

print("FERTILITY -- tokens per word, on text unlike the training corpus:\n")
samples = {
    "in-distribution English": "the cat sat on the mat",
    "unusual English":         "quixotic zealots vex jaded gnomes",
    "German":                  "der Kater sass auf der Matte",
    "Turkish":                 "kedi paspasin ustunde oturdu",
}
print(f"{'text':26} {'words':>6} {'tokens':>7} {'fertility':>10}")
for name, text in samples.items():
    ids = encode(text, merges)
    n_words = len(text.split())
    print(f"{name:26} {n_words:6d} {len(ids):7d} {len(ids)/n_words:10.2f}")

print("\n(These are large because this is a 28-merge toy; a real 100k-vocabulary")
print("tokenizer compresses far better. The RATIOS between languages are the point,")
print("and those survive at production scale -- typically 2-5x, not 1x.)")
print("\nFertility is the fairness metric AND the cost metric: a user whose language has")
print("fertility 4 pays four times as much per word, waits longer, and fits a quarter")
print("as much context -- for the same sentence. It is also worse quality, because the")
print("model must reconstruct meaning from more, less-informative pieces.")

NUMBERS -- how a number splits depends on whether it was frequent in training:

    2020 -> 2 token(s): ['20', '20']
    2021 -> 3 token(s): ['20', '2', '1']
    2023 -> 3 token(s): ['20', '2', '3']
     100 -> 2 token(s): ['1', '00']
     137 -> 3 token(s): ['1', '3', '7']
    4271 -> 4 token(s): ['4', '2', '7', '1']

'2020' is one token because it was frequent; '2023' was never seen and splits
differently. So '2020' and '2023' have completely unrelated representations, and
the model cannot see that they are adjacent years.

This is why arithmetic is hard for LLMs and why several modern tokenizers force
digits to split individually (or in fixed 3-digit groups): a CONSISTENT
decomposition is worth more than a short one.

FERTILITY -- tokens per word, on text unlike the training corpus:

text                        words  tokens  fertility
in-distribution English         6       7       1.17
unusual English                 5      31       6.20
German                          6      20  

## 6. Gotchas & Pitfalls

- **Assuming token count is proportional to character count.** It is not, and the ratio
  varies by language, domain and formatting. Always measure on *your* data.
- **Ignoring pre-tokenization.** The regex that splits before BPE decides whether merges
  can cross whitespace and punctuation. GPT-2's handling of leading spaces (`" cat"` as
  one token) is a pre-tokenization decision, and getting it wrong doubles your vocabulary
  with near-duplicate tokens.
- **Training the tokenizer on a different distribution from the model.** The tokenizer
  should be trained on the actual pre-training mix, including code and non-English at
  their real proportions.
- **Adding tokens after pre-training.** New rows are randomly initialised in a trained
  model. Possible to patch — initialise from the mean embedding, or from the average of
  the pieces it replaces — but never as good as getting it right.
- **Digit merging.** Example 4. If you care about arithmetic, split digits consistently.
- **Forgetting that whitespace is data.** In Python, indentation carries meaning. A
  tokenizer that collapses runs of spaces destroys it; several code models add explicit
  multi-space tokens for exactly this.
- **Byte fallback vs unknown tokens.** Byte-level BPE has no `<unk>` and cannot fail.
  Character-level vocabularies over Unicode can, and will, on emoji and rare scripts.
- **Comparing model costs by characters.** Two providers with different tokenizers charge
  different token counts for identical text.
- **Trusting `len(text.split())` as a token estimate.** Off by 30–100% for English and
  much more elsewhere.

## 7. When to Use vs Alternatives

| Situation | Reach for |
|---|---|
| Training a new LLM from scratch | **Byte-level BPE** (`tiktoken`, `tokenizers`) — the default for good reasons |
| Multilingual, morphologically rich languages | **Unigram LM** via SentencePiece — often lower fertility than BPE |
| You need robustness to segmentation noise | **Subword regularisation** — sample segmentations during training |
| Extreme domain shift (DNA, music, protein) | Train a domain tokenizer; a natural-language one will shred your data |
| Fine-tuning an existing model | **The existing tokenizer, unchanged.** Adding tokens is a last resort |
| Avoiding tokenization entirely | Byte-level models (MegaByte, Mamba-style). Longer sequences, no tokenizer failure modes — an active research direction, not yet the default |

**The honest position.** Tokenization is a solved-enough problem that almost everyone
should use an existing implementation — `tiktoken` for speed, Hugging Face `tokenizers`
for training a new one, `sentencepiece` for the Unigram algorithm. The value in
understanding it is not to write your own; it is to **diagnose** the surprising things it
causes: why the model cannot do arithmetic, why the non-English users complain, why the
code model mangles indentation.

The one decision genuinely worth agonising over is **vocabulary size and composition**,
because it is frozen for the life of the model and it is where the cost and fairness
consequences live.

## 8. Resources

- [Neural Machine Translation of Rare Words with Subword Units](https://arxiv.org/abs/1508.07909) — Sennrich et al.; the paper that brought BPE to NLP.
- [Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) — GPT-2; Section 2.2 introduces byte-level BPE and the pre-tokenization regex.
- [SentencePiece: A simple and language independent subword tokenizer](https://arxiv.org/abs/1808.06226) — the Unigram alternative, and why language-independence matters.
- [Subword Regularization](https://arxiv.org/abs/1804.10959) — Kudo; sampling segmentations as a training-time regulariser.
- [tiktoken](https://github.com/openai/tiktoken) — the fast BPE implementation, with the merge-ranking encode loop of Example 2 done properly.
- [Hugging Face Tokenizers](https://huggingface.co/docs/tokenizers/index) — for actually training one; the pipeline abstraction maps exactly onto the stages here.
- [Language Model Tokenizers Introduce Unfairness Between Languages](https://arxiv.org/abs/2305.15425) — the fertility disparity of Example 4, measured across :many languages.